# Finetune VisDrone — **một model, một tab**Notebook này train **đúng một model**. Mở 4 tab Colab, mỗi tab chạy notebooknày, **chỉ đổi một dòng duy nhất** ở cell 1:```pythonMODEL = "v8n-base"     # tab 1MODEL = "v11n-base"    # tab 2MODEL = "v26n-base"    # tab 3MODEL = "v26n-p2"      # tab 4```Toàn bộ phần còn lại giữ nguyên, không sửa gì. Mỗi tab một GPU, mỗi tab lưuvào thư mục Drive riêng nên không đè lên nhau. Cell cuối gom kết quả từ mọimodel đã xong thành một bảng.> **Colab giới hạn số session chạy cùng lúc** (Pro thường 2–3). Mở 4 tab mà bị> từ chối thì chạy 2 tab trước, xong rồi chạy 2 tab sau — kết quả gom vẫn đủ.---## Bốn điều đã kiểm chứng trên `ultralytics==8.4.118`Không phải suy đoán; mỗi dòng dưới đây đều đo được, và mỗi dòng đều đổi mộtchỗ trong notebook.### 1. `yolo26-p2.yaml` có sẵn trong package — không cần tảiFile trong `ultralytics==8.4.118` khớp đúng bản trên GitHub main:`end2end: True`, `reg_max: 1`, `Detect(P2, P3, P4, P5)` tại `[19, 22, 25, 28]`,scale `n` → **2.662.400 tham số** (dựng thử ra đúng 2.66M).### 2. YOLO26 **không dùng NMS** — output khác hẳn| Model | Output thô | NMS ||---|---|---|| `yolov8n`, `yolo11n` | `(1, 4+nc, 8400)` | cần || **`yolo26n`, `yolo26n-p2`** | **`(1, 300, 6)`** | **không** |`(1, 300, 6)` là box decode sẵn `[x1, y1, x2, y2, conf, cls]`.**Lợi:** pipeline hiện tại tốn **18 ms/frame** NMS trên CPU, so với 47 msinference — bỏ được là khoản cắt lớn nhất còn lại.**Phải xử lý:** cắm v26 vào `3-pipeline/detector.py` sẽ **sai thầm lặng**,không lỗi. Cell export ghi rõ shape từng model.### 3. Đầu end-to-end cắt cứng ở **300** detectionGiao thức VisDrone chấm ở `maxDets=500`, mà val của bạn trung bình **345det/ảnh** ở conf 0.001. Để mặc định thì v26 bị thiệt mà không có dấu hiệu gì.Notebook nâng `max_det = 500`, đã verify output đổi thành `(1, 500, 6)`.### 4. `yolo26n-p2.pt` **không tồn tại** — chỉ 40% trọng số nạp đượcPhải dựng từ yaml rồi nạp một phần từ `yolo26n.pt`. Đo thật, bằng cách tảipretrained về rồi đếm tensor trùng tên **và** trùng shape:| Model | Trọng số nạp được ||---|---|| `v8n-base` | 355/355 — **100%** || `v11n-base` | 499/499 — **100%** || `v26n-base` | 708/708 — **100%** || **`v26n-p2`** | **360/902 — 40%** |Tức **60% model p2 khởi tạo ngẫu nhiên**, trong khi ba model kia nạp đủ. Nênp2 mặc định được **1.5× epoch**, và tỉ lệ này được ghi vào `summary.json` rồilên bảng so sánh. Thiếu con số đó, bảng sẽ bị đọc thành "kiến trúc p2 kém hơn",trong khi thực ra nó chỉ xuất phát sau.Thêm: p2 có stride `[4, 8, 16, 32]` thay vì `[8, 16, 32]` → **34.000 anchor**thay vì 8.400 ở 640px (đo được). Đó là lý do batch mặc định của p2 thấp hơn —và cũng là lý do nó đáng thử với VisDrone, nơi vật thể rất nhỏ.

## 1. Chọn model — **dòng duy nhất cần sửa giữa các tab**

In [ ]:
# ============================================================#  DOI DUNG DONG NAY O MOI TAB. Khong sua gi khac.# ============================================================MODEL = "v8n-base"        # "v8n-base" | "v11n-base" | "v26n-base" | "v26n-p2"# ============================================================SEED = 0IMGSZ = 640EPOCHS_BASE = 100PATIENCE = 30# maxDets cua giao thuc VisDrone. Dau end2end mac dinh cat o 300, ma anh# VisDrone dong co the vuot 300 vat the -> khong nang len la v26 bi thiet# ma khong bao gi.MAX_DET = 500REGISTRY = {    "v8n-base":  dict(cfg="yolov8n.yaml",    weights="yolov8n.pt",                      batch=128, epochs=EPOCHS_BASE),    "v11n-base": dict(cfg="yolo11n.yaml",    weights="yolo11n.pt",                      batch=128, epochs=EPOCHS_BASE),    "v26n-base": dict(cfg="yolo26n.yaml",    weights="yolo26n.pt",                      batch=128, epochs=EPOCHS_BASE),    # Khong co yolo26n-p2.pt -> dung tu yaml, nap mot phan tu yolo26n.pt.    # Them tang P2 (stride 4) -> 34000 anchor thay vi 8400 -> batch thap hon,    # va can nhieu epoch hon vi ~50% trong so khoi tao ngau nhien.    "v26n-p2":   dict(cfg="yolo26n-p2.yaml", weights="yolo26n.pt",                      batch=64,  epochs=int(EPOCHS_BASE * 1.5)),}assert MODEL in REGISTRY, f"MODEL phai la mot trong {list(REGISTRY)}"SPEC = dict(REGISTRY[MODEL])print(f"Tab nay train : {MODEL}")print(f"  cfg      : {SPEC['cfg']}")print(f"  weights  : {SPEC['weights']}")print(f"  batch    : {SPEC['batch']} (cell 9 se do lai va tu chinh)")print(f"  epochs   : {SPEC['epochs']}")print(f"  max_det  : {MAX_DET}")

## 2. Kiểm tra GPU

In [ ]:
import subprocessprint(subprocess.run(    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",     "--format=csv,noheader"],    capture_output=True, text=True).stdout or "!! khong thay nvidia-smi")import torchif torch.cuda.device_count() == 0:    raise SystemExit("Khong co GPU. Runtime > Change runtime type > GPU (A100).")GPU_NAME = torch.cuda.get_device_name(0)VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9print(f"\nGPU  : {GPU_NAME}")print(f"VRAM : {VRAM_GB:.1f} GB")if "A100" not in GPU_NAME:    print(f"\n[canh bao] Khong phai A100. Batch mac dinh tinh cho A100 40GB; "          f"cell 9 se do lai va ha xuong cho vua {VRAM_GB:.0f} GB.")

## 3. Cài đặtGhim đúng version đã kiểm chứng. `yolo26` chỉ có từ ultralytics 8.4.x — bản cũhơn báo "model not found" cho hai model v26.

In [ ]:
%pip install -q "ultralytics==8.4.118" onnx onnxslim onnxruntimeimport ultralytics, torch, platformprint("ultralytics", ultralytics.__version__)print("torch      ", torch.__version__, "| cuda", torch.version.cuda)print("python     ", platform.python_version())from ultralytics.utils.downloads import GITHUB_ASSETS_NAMESw = SPEC["weights"]print(f"\n{w}: {'co pretrained' if w in GITHUB_ASSETS_NAMES else 'KHONG co'}")if MODEL == "v26n-p2":    print("  (dung: yolo26n-p2.pt khong ton tai. Dung yolo26n.pt nap mot phan.)")

## 4. Mount Drive**Trước khi chạy**: mở link dataset → **Add shortcut to Drive → My Drive**.Thư mục chia sẻ không tự xuất hiện trong `MyDrive` nếu chưa tạo shortcut, và`gdown --folder` bị chặn ở 50 file nên vô dụng với dataset vài nghìn ảnh.

In [ ]:
from google.colab import drivedrive.mount("/content/drive")import osROOT = "/content/drive/MyDrive"print("Thu muc cap 1 trong MyDrive:\n")for d in sorted(os.listdir(ROOT))[:60]:    if os.path.isdir(os.path.join(ROOT, d)):        print("  ", d)

## 5. Soi datasetĐiền `DATASET_DIR` (lấy tên từ cell 4). Cell này không train gì — nó chỉ chobạn thấy dataset thực sự chứa gì. Mọi lỗi dataset đều rẻ khi phát hiện ở đâyvà rất đắt khi phát hiện sau 3 tiếng.

In [ ]:
import os, glob# <<< SUA DONG NAY (giong nhau o ca 4 tab) >>>DATASET_DIR = "/content/drive/MyDrive/TEN_THU_MUC_DATASET"assert os.path.isdir(DATASET_DIR), (    f"Khong thay {DATASET_DIR}. Kiem tra ten o cell 4, "    "va da 'Add shortcut to Drive' chua.")IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".JPG", ".JPEG", ".PNG")def tree(root, depth=0, maxdepth=2):    if depth > maxdepth:        return    for e in sorted(os.listdir(root))[:25]:        p = os.path.join(root, e)        if os.path.isdir(p):            try:                fs = os.listdir(p)            except OSError:                fs = []            n_img = sum(1 for f in fs if f.endswith(IMG_EXT))            n_txt = sum(1 for f in fs if f.endswith(".txt"))            tag = ", ".join([x for x in (f"{n_img} anh" if n_img else "",                                         f"{n_txt} txt" if n_txt else "") if x])            print("  " * depth + f"[{e}]" + (f"  <- {tag}" if tag else ""))            tree(p, depth + 1, maxdepth)        elif depth == 0:            print("  " * depth + e)print(f"Cau truc {DATASET_DIR}:\n")tree(DATASET_DIR)archives = [p for p in glob.glob(os.path.join(DATASET_DIR, "**", "*"), recursive=True)            if p.endswith((".zip", ".tar", ".tar.gz", ".tgz"))]if archives:    print("\nFile nen (stage se nhanh hon nhieu, dien vao ARCHIVE o cell 6):")    for a in archives[:10]:        print(f"  {a}   {os.path.getsize(a)/1e9:.2f} GB")for y in glob.glob(os.path.join(DATASET_DIR, "**", "*.yaml"), recursive=True)[:3]:    print(f"\n--- {y} ---")    print(open(y).read()[:600])

## 6. Chép dataset về đĩa local**Đừng train trực tiếp trên Drive.** Drive gắn qua FUSE — mỗi lần mở một fileảnh là một lượt gọi mạng. Train ở batch 128 cần vài trăm ảnh/giây, FUSE khôngđáp ứng nổi; GPU sẽ ngồi chờ và bạn sẽ tưởng model chậm trong khi thật ra làI/O.Mỗi tab là một runtime riêng nên **tab nào cũng phải chép lại**. Nếu dataset cósẵn file `.zip`, điền vào `ARCHIVE` — chép một file lớn qua FUSE nhanh hơnhàng nghìn file nhỏ rất nhiều.

In [ ]:
import os, time, shutil, subprocess, globLOCAL = "/content/dataset"os.makedirs(LOCAL, exist_ok=True)ARCHIVE = ""   # vd "/content/drive/MyDrive/xxx/visdrone.zip"; de trong = chep ca cay thu muct0 = time.time()if ARCHIVE:    print(f"Chep {os.path.basename(ARCHIVE)} ...")    local_arc = os.path.join("/content", os.path.basename(ARCHIVE))    shutil.copy(ARCHIVE, local_arc)    print(f"  xong sau {time.time()-t0:.0f}s, giai nen ...")    shutil.unpack_archive(local_arc, LOCAL)    os.remove(local_arc)else:    print("Chep ca cay thu muc (cham hon) ...")    subprocess.run(["cp", "-r", DATASET_DIR + "/.", LOCAL], check=True)n_img = sum(1 for p in glob.glob(os.path.join(LOCAL, "**", "*"), recursive=True)            if p.endswith(IMG_EXT))sz = subprocess.run(["du", "-sh", LOCAL], capture_output=True, text=True).stdout.split()[0]print(f"\n[ok] {n_img} anh, {sz}, mat {(time.time()-t0)/60:.1f} phut -> {LOCAL}")

## 7. Dựng và **kiểm tra** `data.yaml`Bốn kiểm tra, mỗi cái ứng với một kiểu hỏng rất khó truy về sau:| Kiểm tra | Bỏ qua thì sao ||---|---|| Ảnh có label khớp | Ảnh không label bị coi là "không có vật thể" → model học cách bỏ sót || Toạ độ trong `[0,1]` | Label pixel thay vì chuẩn hoá → loss vẫn giảm, mAP vẫn ~0 || `class id < nc` | Index out of range giữa chừng epoch || Histogram lớp | Lớp rỗng → AP lớp đó = −1, kéo lệch mAP |

In [ ]:
import os, glob, yaml, random, collectionsdef find_split(root, split):    for pat in (f"{split}/images", f"images/{split}", f"{split}",                f"*{split}*/images", f"images/*{split}*"):        for h in glob.glob(os.path.join(root, pat)):            if os.path.isdir(h) and any(f.endswith(IMG_EXT) for f in os.listdir(h)):                return h    return Nonesplits = {s: find_split(LOCAL, s) for s in ("train", "val", "test")}for s, p in splits.items():    n = len([f for f in os.listdir(p) if f.endswith(IMG_EXT)]) if p else 0    print(f"  {s:6s} {p or '(khong thay)'}  {n} anh")assert splits["train"], "Khong thay split train. Sua tay bien `splits` roi chay lai."assert splits["val"], "Khong co val -> khong co so lieu de bao cao."def label_dir_for(img_dir):    for c in (img_dir.replace("/images", "/labels"),              os.path.join(os.path.dirname(img_dir), "labels")):        if os.path.isdir(c):            return c    return Nonenames = Nonefor y in glob.glob(os.path.join(LOCAL, "**", "*.yaml"), recursive=True):    d = yaml.safe_load(open(y))    if isinstance(d, dict) and "names" in d:        names = d["names"]        print(f"\n[ok] ten lop lay tu {y}")        breakif names is None:    print("\n[canh bao] khong co data.yaml -> dung ten lop VisDrone mac dinh")    names = ["pedestrian", "people", "bicycle", "car", "van", "truck",             "tricycle", "awning-tricycle", "bus", "motor"]if isinstance(names, dict):    names = [names[k] for k in sorted(names)]NC = len(names)print(f"[ok] nc = {NC}: {names}")problems, hist = [], collections.Counter()for split, img_dir in splits.items():    if not img_dir:        continue    lab_dir = label_dir_for(img_dir)    if not lab_dir:        problems.append(f"{split}: khong thay thu muc labels")        continue    imgs = [f for f in os.listdir(img_dir) if f.endswith(IMG_EXT)]    missing = sum(1 for f in imgs                  if not os.path.exists(                      os.path.join(lab_dir, os.path.splitext(f)[0] + ".txt")))    if missing:        problems.append(f"{split}: {missing}/{len(imgs)} anh khong co file label")    lfs = glob.glob(os.path.join(lab_dir, "*.txt"))    bad_range = bad_cls = n_box = 0    for lf in random.Random(0).sample(lfs, min(400, len(lfs))):        for line in open(lf):            parts = line.split()            if len(parts) < 5:                continue            n_box += 1            c = int(float(parts[0]))            hist[c] += 1            if not (0 <= c < NC):                bad_cls += 1            if any(not (0.0 <= float(v) <= 1.0) for v in parts[1:5]):                bad_range += 1    if bad_range:        problems.append(f"{split}: {bad_range}/{n_box} box ngoai [0,1] "                        f"-> label chua chuan hoa")    if bad_cls:        problems.append(f"{split}: {bad_cls}/{n_box} box co class id ngoai [0,{NC})")print("\nPhan bo lop (mau 400 file/split):")mx = max(hist.values()) if hist else 1for c in range(NC):    print(f"  {c:2d} {names[c]:18s} {hist[c]:7d} {'#' * int(40 * hist[c] / mx)}")    if hist[c] == 0:        problems.append(f"lop {c} ({names[c]}) khong co box nao -> AP lop nay = -1")DATA_YAML = "/content/data.yaml"cfg = {"path": LOCAL, "train": splits["train"], "val": splits["val"],       "nc": NC, "names": names}if splits["test"]:    cfg["test"] = splits["test"]yaml.safe_dump(cfg, open(DATA_YAML, "w"), sort_keys=False, allow_unicode=True)print("\n" + "=" * 62)if problems:    print("VAN DE - doc ky truoc khi train:")    for p in problems:        print("  !!", p)else:    print("Khong phat hien van de nao.")print("=" * 62)print(open(DATA_YAML).read())

## 8. Preflight — dựng model trước khi trainBa tiếng train rồi mới biết model không dựng được là ba tiếng mất trắng. Cellnày dựng model, in số tham số, số anchor, và **tỉ lệ trọng số nạp được** —con số cuối chính là bằng chứng cho caveat của p2, nên nó phải hiện ra chứkhông được im lặng.

In [ ]:
import warnings, io, contextlib, torchwarnings.filterwarnings("ignore")from ultralytics import YOLObuf = io.StringIO()with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):    _m = YOLO(SPEC["cfg"], verbose=False)    _m.load(SPEC["weights"])    _src = YOLO(SPEC["weights"]).model.state_dict()# Dem truc tiep tren state_dict thay vi doc log: ultralytics in dong# "Transferred x/y" qua LOGGER rieng, redirect_stdout khong bat duoc -- va mot# cot im lang bao "n/a" chinh la cot khong ai kiem tra. Day dung la tieu chi# ultralytics dung khi nap: trung ten VA trung shape._dst = _m.model.state_dict()MATCHED = sum(1 for k, v in _dst.items()              if k in _src and _src[k].shape == v.shape)TRANSFER = f"{MATCHED}/{len(_dst)}"TRANSFER_FRAC = MATCHED / len(_dst)_head = _m.model.model[-1]END2END = bool(getattr(_head, "end2end", False))if END2END:    _head.max_det = MAX_DET_m.model.eval()with torch.no_grad():    _y = _m.model(torch.zeros(1, 3, IMGSZ, IMGSZ))_out = _y[0] if isinstance(_y, (list, tuple)) else _yN_PARAMS = sum(p.numel() for p in _m.model.parameters())N_ANCHORS = None if END2END else int(_out.shape[-1])# Preflight chay o nc=80 mac dinh cua yaml (model chua gap dataset), nen shape# do duoc bay gio khong phai shape cuoi. Ghi ca hai de khong ai doc nham.OUT_SHAPE_TRAINED = (1, MAX_DET, 6) if END2END else (1, 4 + NC, N_ANCHORS)print(f"model            {MODEL}")print(f"tham so          {N_PARAMS/1e6:.2f} M")print(f"anchor           {N_ANCHORS if N_ANCHORS else '(end2end, khong dung anchor grid)'}")print(f"NMS              {'KHONG can (end-to-end)' if END2END else 'can'}")print(f"max_det          {MAX_DET if END2END else '(NMS quyet dinh)'}")print(f"weights nap      {TRANSFER}  ({TRANSFER_FRAC*100:.0f}%)")print(f"output sau train {OUT_SHAPE_TRAINED}")print(f"  (preflight do duoc {tuple(_out.shape)} vi chay o nc=80 mac dinh cua yaml;")print(f"   shape that duoc ghi lai tu file ONNX o cell 12)")# Nguong 0.75 chu khong phai 0.9: model base thuong chi dat ~90% vi dau detect# khong khop khi nc khac COCO -- do la binh thuong. Chi truong hop mat ca# backbone/neck moi dang canh bao.if TRANSFER_FRAC < 0.75:    print(f"\n[luu y] Mot phan lon model khoi tao ngau nhien ({TRANSFER}).")    print(f"        Da bu bang {SPEC['epochs']} epoch (cao hon {EPOCHS_BASE} cua cac model kia),")    print(f"        nhung bang so sanh cuoi VAN phai ghi ro con so nay -- neu khong,")    print(f"        nguoi doc se ket luan sai ve kien truc.")del _m, _src, _dsttorch.cuda.empty_cache()

## 9. Đo bộ nhớ thật rồi tự chỉnh batch`batch` ở cell 1 là ước lượng cho A100 40 GB. Cell này **đo thật** — vài bướcforward + backward với dữ liệu ngẫu nhiên rồi đọc đỉnh bộ nhớ — và tự **tăng**nếu còn nhiều chỗ trống hoặc **giảm** nếu tràn.Đây là cận dưới (chưa tính EMA và optimizer state, vốn nhỏ với model nano),nhưng nó bắt OOM ở phút thứ hai thay vì giờ thứ hai.

In [ ]:
import torch, gc, warningswarnings.filterwarnings("ignore")from ultralytics import YOLOdef _all_tensors(o):    """Dau train tra ve dict (yolo26 tra one2many/one2one), khong phai list.    Gom het tensor lai roi tinh mot loss gia -- chi de do bo nho."""    if torch.is_tensor(o):        return [o]    if isinstance(o, dict):        o = list(o.values())    if isinstance(o, (list, tuple)):        out = []        for x in o:            out.extend(_all_tensors(x))        return out    return []def peak_mem_gb(cfg, batch, imgsz=IMGSZ, steps=3):    torch.cuda.empty_cache(); gc.collect()    torch.cuda.reset_peak_memory_stats()    m = YOLO(cfg, verbose=False).model.cuda().train()    opt = torch.optim.SGD(m.parameters(), lr=1e-4)    scaler = torch.amp.GradScaler("cuda")    try:        for _ in range(steps):            x = torch.rand(batch, 3, imgsz, imgsz, device="cuda")            with torch.amp.autocast("cuda"):                ts = [t for t in _all_tensors(m(x)) if t.is_floating_point()]                if not ts:                    raise RuntimeError("khong lay duoc tensor nao tu dau ra")                loss = sum((t.float() ** 2).mean() for t in ts)            scaler.scale(loss).backward()            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)        peak = torch.cuda.max_memory_allocated() / 1e9    except torch.cuda.OutOfMemoryError:        peak = float("inf")    finally:        del m, opt        torch.cuda.empty_cache(); gc.collect()    return peakBUDGET = VRAM_GB * 0.85     # chua 15% cho fragmentation va cudnn workspaceprint(f"VRAM {VRAM_GB:.1f} GB -> ngan sach {BUDGET:.1f} GB\n")batch = SPEC["batch"]peak = peak_mem_gb(SPEC["cfg"], batch)print(f"  batch {batch:4d} -> dinh {peak:5.1f} GB")while peak > BUDGET and batch > 8:    batch = max(8, batch // 2)    peak = peak_mem_gb(SPEC["cfg"], batch)    print(f"  ha xuong {batch:4d} -> dinh {peak:5.1f} GB")while peak * 1.9 < BUDGET and batch < 512:    trial = batch * 2    p2 = peak_mem_gb(SPEC["cfg"], trial)    if p2 > BUDGET:        print(f"  thu {trial:4d} -> {p2:5.1f} GB, vuot ngan sach, giu {batch}")        break    batch, peak = trial, p2    print(f"  tang len {batch:4d} -> dinh {peak:5.1f} GB")SPEC["batch"] = batchprint(f"\n[chot] batch = {batch}  (dinh do duoc {peak:.1f} / {BUDGET:.1f} GB)")

## 10. Train`project` trỏ **thẳng vào Drive** là có chủ ý: `/content` bị xoá sạch khi Colabngắt kết nối, nên checkpoint để ở đó thì mất trắng nhiều giờ. Để trên Drive thì`last.pt` sống sót, và chạy lại đúng cell này là **tự resume**.Dataset vẫn nằm ở `/content` (nhanh); chỉ checkpoint đi Drive. Đó là chỗ phânchia đúng: ảnh đọc mỗi bước, checkpoint ghi mỗi epoch.Augment: giữ photometric, **hạn chế geometric mạnh** — vật thể VisDrone rất nhỏ,`scale`/`shear` lớn sẽ xoá sạch các box dưới 10 px.

In [ ]:
import os, warningswarnings.filterwarnings("ignore")from ultralytics import YOLOPROJECT = "/content/drive/MyDrive/skysentry/finetune_runs"os.makedirs(PROJECT, exist_ok=True)last = os.path.join(PROJECT, MODEL, "weights", "last.pt")if os.path.exists(last):    print(f"[resume] tim thay {last}\n")    model = YOLO(last)    resume = Trueelse:    print(f"[moi] dung {SPEC['cfg']} + nap {SPEC['weights']}\n")    model = YOLO(SPEC["cfg"], verbose=False)    model.load(SPEC["weights"])    resume = Falsehead = model.model.model[-1]if getattr(head, "end2end", False):    head.max_det = MAX_DET    print(f"[e2e] max_det 300 -> {MAX_DET}\n")results = model.train(    data=DATA_YAML, epochs=SPEC["epochs"], imgsz=IMGSZ, batch=SPEC["batch"],    device=0, workers=8, seed=SEED, deterministic=False,    project=PROJECT, name=MODEL, exist_ok=True, resume=resume,    patience=PATIENCE, amp=True, cache=False, val=True, plots=True,    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,    degrees=0.0, translate=0.1, scale=0.5, shear=0.0, perspective=0.0,    flipud=0.0, fliplr=0.5, mosaic=1.0, mixup=0.0, close_mosaic=10,)print("\n[xong] train hoan tat")# Da cham tran epoch hay da dung som? Voi p2 (chi nap 40% pretrained) day la# cau hoi quan trong: cham tran nghia la mAP van con dang len khi het epoch,# tuc con so cuoi cung la mot gioi han nhan tao, khong phai gioi han kien truc.import pandas as pd_csv = os.path.join(PROJECT, MODEL, "results.csv")if os.path.exists(_csv):    _d = pd.read_csv(_csv)    _d.columns = [c.strip() for c in _d.columns]    _n = len(_d)    _col = next((c for c in _d.columns if "mAP50-95" in c), None)    if _col:        _best_ep = int(_d[_col].idxmax()) + 1        print(f"  epoch chay     : {_n}/{SPEC['epochs']}")        print(f"  epoch tot nhat : {_best_ep}")        if _n >= SPEC["epochs"] and _best_ep > SPEC["epochs"] - PATIENCE:            print(f"\n  [luu y] Cham tran epoch va epoch tot nhat nam o cuoi -> mAP CON DANG LEN.")            print(f"          Con so nay la gioi han cua so epoch, khong phai cua kien truc.")            print(f"          Muon ket luan cong bang thi tang epochs roi chay lai cell nay (tu resume).")        else:            print(f"  -> da dung som / da bao hoa, con so nay dung la gioi han kien truc.")

## 11. Validate theo giao thức VisDrone`max_det=500`, không phải 300 mặc định. Ghi ra `summary.json` để cell gom kếtquả đọc được.

In [ ]:
import json, osfrom ultralytics import YOLObest = os.path.join(PROJECT, MODEL, "weights", "best.pt")assert os.path.exists(best), f"khong thay {best}"m = YOLO(best)h = m.model.model[-1]if getattr(h, "end2end", False):    h.max_det = MAX_DETres = m.val(data=DATA_YAML, imgsz=IMGSZ, batch=SPEC["batch"], device=0,            max_det=MAX_DET, verbose=False)summary = {    "model": MODEL,    "mAP50-95": float(res.box.map),    "mAP50": float(res.box.map50),    "mAP75": float(res.box.map75),    "ap_per_class": {names[i]: float(v) for i, v in enumerate(res.box.maps)},    "params_M": round(N_PARAMS / 1e6, 3),    "anchors": N_ANCHORS,    "end2end_no_nms": END2END,    "max_det": MAX_DET,    "weights_transferred": TRANSFER,    "transfer_frac": round(TRANSFER_FRAC, 4),    "epochs": SPEC["epochs"], "batch": SPEC["batch"], "imgsz": IMGSZ,    "seed": SEED, "nc": NC,    "output_shape_expected": list(OUT_SHAPE_TRAINED),    "gpu": GPU_NAME,    "ultralytics": ultralytics.__version__,}json.dump(summary, open(os.path.join(PROJECT, MODEL, "summary.json"), "w"),          indent=2, ensure_ascii=False)print(f"{MODEL}")print(f"  mAP50-95 {summary['mAP50-95']:.4f}")print(f"  mAP50    {summary['mAP50']:.4f}")print(f"  mAP75    {summary['mAP75']:.4f}")print("\nAP theo lop:")for k, v in summary["ap_per_class"].items():    print(f"  {k:18s} {v:.4f}")

## 12. Export ONNX cho pipeline QCS8550Đúng thiết lập board yêu cầu, và vá sẵn lỗi đã gặp thật:- `opset=13` — opset mới sinh op mà QNN đẩy ngược về CPU- `dynamic=False`, shape tĩnh — bắt buộc để tạo QNN context binary- **`sanitise_onnx()`** — Ultralytics + onnxslim để tensor output nằm cả trong  `graph.output` lẫn `value_info`. ONNX Runtime bỏ qua, còn AI Hub **từ chối  compile**: `Tensors {'output0'} occur in value_info but also in model IO`.  Lỗi này đã chặn pipeline một lần rồi.- `nms=False` chỉ truyền cho model cần NMS. Truyền cho model end2end là vô  nghĩa và có thể làm export lỗi.

In [ ]:
import os, json, hashlib, shutil, warningswarnings.filterwarnings("ignore")from ultralytics import YOLOdef sha256(path, chunk=1 << 20):    h = hashlib.sha256()    with open(path, "rb") as f:        for b in iter(lambda: f.read(chunk), b""):            h.update(b)    return h.hexdigest()def sanitise_onnx(path):    """Bo tensor vua nam trong graph IO vua nam trong value_info."""    import onnx    mo = onnx.load(path)    io_names = {t.name for t in mo.graph.input} | {t.name for t in mo.graph.output}    dupes = [vi.name for vi in mo.graph.value_info if vi.name in io_names]    if dupes:        keep = [vi for vi in mo.graph.value_info if vi.name not in io_names]        del mo.graph.value_info[:]        mo.graph.value_info.extend(keep)        onnx.save(mo, path)    return dupesOUT = os.path.join(PROJECT, MODEL)m = YOLO(best)h = m.model.model[-1]if getattr(h, "end2end", False):    h.max_det = MAX_DETkw = dict(format="onnx", imgsz=IMGSZ, opset=13, dynamic=False,          simplify=True, batch=1)if not END2END:    kw["nms"] = Falsep = m.export(**kw)onnx_path = os.path.join(OUT, f"{MODEL}.onnx")shutil.move(str(p), onnx_path)dupes = sanitise_onnx(onnx_path)import onnxruntime as ortsess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])ishape = sess.get_inputs()[0].shapeoshape = sess.get_outputs()[0].shapemeta = {    "model": MODEL,    "onnx_sha256_16": sha256(onnx_path)[:16],    "pt_sha256_16": sha256(best)[:16],    "input_shape": ishape, "output_shape": oshape,    "end2end_no_nms": END2END, "max_det": MAX_DET if END2END else None,    "opset": 13, "imgsz": IMGSZ, "nc": NC,    "value_info_dupes_removed": len(dupes),    "pipeline_compatible": not END2END,}json.dump(meta, open(os.path.join(OUT, "export.json"), "w"), indent=2)print(f"[ok] {onnx_path}")print(f"     input  {ishape}")print(f"     output {oshape}")print(f"     sha256 {meta['onnx_sha256_16']}")if dupes:    print(f"     da va {len(dupes)} value_info trung (AI Hub se tu choi neu khong va)")print("\n" + "=" * 64)if END2END:    print("KHONG tuong thich truc tiep voi 3-pipeline/detector.py")    print(f"  output {oshape} = box da decode san [x1,y1,x2,y2,conf,cls]")    print("  detector.py dang decode dang (1, 4+nc, A) -> cam vao se SAI THAM LANG.")    print("  Can them nhanh: neu shape[-1]==6 thi bo NMS, chi loc theo conf,")    print("  roi dua toa do ve he anh goc bang gain/pad nhu cu.")else:    print("Tuong thich voi 3-pipeline/detector.py")    print(f"  output {oshape} - decode nhu cu, NMS chay tren Kryo va do rieng.")print("=" * 64)

## 13. Gom kết quả từ mọi tabChạy cell này ở **bất kỳ tab nào sau khi các tab khác xong**. Nó đọc`summary.json` của mọi model đã train xong trong cùng thư mục Drive.Cột `weights nạp` phải có mặt: thiếu nó, bảng này sẽ bị đọc thành "kiến trúc p2kém hơn", trong khi thực ra p2 chỉ xuất phát sau.

In [ ]:
import json, glob, osimport pandas as pdrows = []for sp in sorted(glob.glob(os.path.join(PROJECT, "*", "summary.json"))):    s = json.load(open(sp))    ep = os.path.join(os.path.dirname(sp), "export.json")    e = json.load(open(ep)) if os.path.exists(ep) else {}    rows.append({        "model": s["model"],        "mAP50-95": round(s["mAP50-95"], 4),        "mAP50": round(s["mAP50"], 4),        "mAP75": round(s["mAP75"], 4),        "params(M)": s["params_M"],        "anchors": s["anchors"] or "e2e",        "NMS": "khong" if s["end2end_no_nms"] else "can",        "max_det": s["max_det"],        "epochs": s["epochs"],        "batch": s["batch"],        "weights nap": s["weights_transferred"],        "onnx output": str(e.get("output_shape", "chua export")),    })if not rows:    print("Chua co model nao xong.")else:    df = pd.DataFrame(rows).sort_values("mAP50-95", ascending=False)    display(df)    out_csv = os.path.join(PROJECT, "comparison.csv")    df.to_csv(out_csv, index=False)    print(f"\n[ok] {out_csv}")    print(f"[ok] {len(rows)}/4 model da xong: {', '.join(r['model'] for r in rows)}")    print("\nDoc bang nay can nho:")    print("  - Cot 'weights nap': model nao thap hon ~75% la xuat phat sau,")    print("    khong phai kien truc kem hon.")    print("  - Cot 'onnx output': dang (1,N,6) la box decode san, KHONG chay NMS;")    print("    dang (1,4+nc,A) moi cam thang vao pipeline hien tai duoc.")

---## Sau khi cả 4 model xong**Bắt buộc trước khi dùng v26 trong pipeline.** `3-pipeline/detector.py` hiệnchỉ decode `(1, 4+nc, A)`. Hai model v26 trả `(1, N, 6)` đã decode sẵn. Cắm vàomà không sửa thì **không có lỗi nào được báo** — chỉ là kết quả sai. Cần thêmnhánh: `shape[-1] == 6` → tách `[x1, y1, x2, y2, conf, cls]`, bỏ NMS, lọc theo`conf`, rồi đưa toạ độ về hệ ảnh gốc bằng `gain`/`pad` như cũ.**Việc đáng làm nhất sau đó.** Bỏ NMS là bỏ **18 ms/frame** trên CPU — khoảncắt lớn nhất còn lại trong frame budget (inference chỉ 47 ms). Nhưng phải kiểmtra thật: đầu end-to-end dùng `topk`, và `topk` có thể bị QNN đẩy về CPU. Chạycompile job trên AI Hub rồi đọc `n_ops_fallback` **trước khi** tin vào con sốnày — nếu > 0 thì phần tiết kiệm sẽ bị trả lại ở chỗ khác.**Mỗi dòng kết quả phải ghi kèm** `max_det` (500, không phải 300), tỉ lệ trọngsố nạp được, `imgsz`, `seed`, và sha256 của ONNX. Thiếu bất kỳ cái nào là haibảng không so được với nhau.